# SmartCart - Notebook 2: User-Based Collaborative Filtering

This notebook builds a user-based recommender using cosine similarity and evaluates ranking quality with standard top-K metrics.

## Objectives
- Construct a reliable user-item matrix
- Compute user-user cosine similarity
- Generate top-N product recommendations
- Evaluate model quality using Precision@K, Recall@K, MAP@K, Coverage, and Diversity

## Inputs
- `data/processed/user_data_clean.csv`
- `data/processed/product_data_clean.csv`
- `data/processed/user_item_matrix_filled.csv`
- `data/processed/user_category_agg.csv`

This notebook expects Notebook 1 to be run first so processed artifacts are available.

In [1]:
# Imports
from pathlib import Path
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [2]:
# Load preprocessed artifacts generated in Notebook 1
cwd = Path.cwd()
processed_candidates = [cwd / 'data' / 'processed', cwd.parent / 'data' / 'processed']
processed_dir = next((p for p in processed_candidates if p.exists()), processed_candidates[0])
required_files = [
    'user_data_clean.csv',
    'product_data_clean.csv',
    'user_item_matrix_filled.csv',
    'user_category_agg.csv',
]

missing = [f for f in required_files if not (processed_dir / f).exists()]
if missing:
    raise FileNotFoundError(
        'Missing processed files. Run Notebook 1 first. Missing: ' + ', '.join(missing)
    )

user_data = pd.read_csv(processed_dir / 'user_data_clean.csv', parse_dates=['Timestamp'])
product_data = pd.read_csv(processed_dir / 'product_data_clean.csv')
user_item_matrix_filled = pd.read_csv(processed_dir / 'user_item_matrix_filled.csv', index_col=0).astype(np.float64)
user_category_agg = pd.read_csv(processed_dir / 'user_category_agg.csv', parse_dates=['LastInteraction'])

print('Loaded processed artifacts from:', processed_dir)
print('user_data shape:', user_data.shape)
print('user_item_matrix shape:', user_item_matrix_filled.shape)

Loaded processed artifacts from: /Users/solaris003/Repository/Projects/Recommender System & Pattern Mining for E-Commerce Analytics/data/processed
user_data shape: (724, 5)
user_item_matrix shape: (50, 100)


In [3]:
# Similarity utility
def safe_cosine_similarity(matrix_like):
    matrix = np.asarray(matrix_like, dtype=np.float64)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)

    with warnings.catch_warnings():
        warnings.filterwarnings(
            'ignore',
            message='.*encountered in matmul',
            category=RuntimeWarning,
        )
        with np.errstate(divide='ignore', over='ignore', invalid='ignore'):
            return cosine_similarity(matrix)

similarity_matrix = safe_cosine_similarity(user_item_matrix_filled)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_item_matrix_filled.index,
    columns=user_item_matrix_filled.index,
)

similarity_df.head()

UserID,U000,U001,U002,U003,U004,U005,U006,U007,U008,U009,U010,U011,U012,U013,U014,U015,U016,U017,U018,U019,U020,U021,U022,U023,U024,U025,U026,U027,U028,U029,U030,U031,U032,U033,U034,U035,U036,U037,U038,U039,U040,U041,U042,U043,U044,U045,U046,U047,U048,U049
UserID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
U000,1.000000,0.063071,0.195522,0.023466,0.065412,0.161251,0.160096,0.092083,0.238263,0.274844,0.296826,0.105579,0.085622,0.242357,0.030061,0.044501,0.005671,0.149640,0.280168,0.115609,0.127578,0.050850,0.021226,0.159421,0.252215,0.091461,0.158070,0.130966,0.374732,0.253025,0.089830,0.045056,0.101926,0.134746,0.006189,0.101655,0.069422,0.184228,0.171567,0.186954,0.241693,0.129483,0.156790,0.132200,0.161478,0.100346,0.126917,0.150727,0.000000,0.104294
U001,0.063071,1.000000,0.190861,0.000000,0.111332,0.009540,0.000000,0.172286,0.167460,0.017593,0.126938,0.197880,0.339629,0.063088,0.039126,0.223408,0.221416,0.272669,0.214885,0.090283,0.120762,0.000000,0.257848,0.138330,0.190051,0.000000,0.149626,0.288016,0.145686,0.231228,0.041757,0.146608,0.214786,0.083513,0.016111,0.037802,0.175066,0.184975,0.000000,0.019210,0.121540,0.024075,0.097953,0.007821,0.000000,0.000000,0.193670,0.247805,0.000000,0.000000
U002,0.195522,0.190861,1.000000,0.065094,0.111662,0.050830,0.027756,0.055877,0.000000,0.181229,0.211728,0.216727,0.163996,0.016807,0.000000,0.088175,0.078650,0.023720,0.000000,0.025656,0.064345,0.156729,0.009813,0.184262,0.138086,0.000000,0.099655,0.056373,0.047250,0.246406,0.035598,0.018748,0.114442,0.035598,0.008584,0.087282,0.174522,0.146013,0.147309,0.150123,0.144756,0.000000,0.217465,0.000000,0.055992,0.034794,0.194243,0.110030,0.177165,0.000000
U003,0.023466,0.000000,0.065094,1.000000,0.035737,0.104116,0.026650,0.000000,0.025384,0.288009,0.097880,0.000000,0.123072,0.032275,0.080064,0.056440,0.201374,0.091098,0.257618,0.024633,0.298604,0.120386,0.037689,0.150970,0.023570,0.199875,0.017010,0.000000,0.000000,0.028677,0.079752,0.168005,0.146508,0.193683,0.043959,0.103142,0.030817,0.074770,0.072532,0.069886,0.243836,0.000000,0.000000,0.074688,0.197121,0.000000,0.054396,0.126773,0.374228,0.250000
U004,0.065412,0.111332,0.111662,0.035737,1.000000,0.159064,0.057144,0.026294,0.195942,0.247023,0.256695,0.038589,0.116424,0.000000,0.000000,0.072613,0.181352,0.146500,0.182860,0.147894,0.072859,0.290401,0.080814,0.091045,0.000000,0.000000,0.071124,0.144430,0.161201,0.319747,0.131919,0.066902,0.133050,0.219864,0.268633,0.204573,0.133808,0.210424,0.083984,0.050575,0.062741,0.116202,0.078797,0.048044,0.103747,0.133716,0.124969,0.217464,0.255318,0.172729


In [4]:
# Recommendation function
def recommend_top_n(
    target_user,
    ratings_matrix,
    user_similarity_df,
    n=5,
    neighbor_top_m=10,
    min_neighbor_similarity=0.0,
    min_pred_score=0.0,
    already_rated_threshold=0,
    return_scores=False,
):
    if target_user not in ratings_matrix.index:
        return [] if not return_scores else pd.DataFrame(columns=['ProductID', 'PredictedScore'])

    neighbors = (
        user_similarity_df.loc[target_user]
        .drop(index=target_user, errors='ignore')
        .sort_values(ascending=False)
    )
    neighbors = neighbors[neighbors > min_neighbor_similarity].head(neighbor_top_m)

    if neighbors.empty:
        return [] if not return_scores else pd.DataFrame(columns=['ProductID', 'PredictedScore'])

    neighbor_ratings = ratings_matrix.loc[neighbors.index]
    target_ratings = ratings_matrix.loc[target_user]

    unrated_items = target_ratings[target_ratings <= already_rated_threshold].index
    if len(unrated_items) == 0:
        return [] if not return_scores else pd.DataFrame(columns=['ProductID', 'PredictedScore'])

    weighted_sum = neighbor_ratings[unrated_items].T.dot(neighbors.values)
    sim_sum = np.abs(neighbors.values).sum()

    if sim_sum == 0:
        return [] if not return_scores else pd.DataFrame(columns=['ProductID', 'PredictedScore'])

    pred_scores = (weighted_sum / sim_sum).sort_values(ascending=False)
    pred_scores = pred_scores[pred_scores > min_pred_score]

    if return_scores:
        return pred_scores.head(n).rename('PredictedScore').reset_index().rename(columns={'index': 'ProductID'})

    return pred_scores.head(n).index.tolist()

In [5]:
# Example recommendation for one user
sample_user = user_item_matrix_filled.index[0]
sample_recommendations = recommend_top_n(
    target_user=sample_user,
    ratings_matrix=user_item_matrix_filled,
    user_similarity_df=similarity_df,
    n=5,
    neighbor_top_m=10,
    min_neighbor_similarity=0.0,
    return_scores=True,
)

print(f'Sample user: {sample_user}')
sample_recommendations

Sample user: U000


,ProductID,PredictedScore
0,P0064,1.447579
1,P0052,1.440173
2,P0083,1.373935
3,P0051,1.246704
4,P0088,1.228732


In [6]:
# Evaluation: Precision@K, Recall@K, MAP@K, Coverage, Diversity
K = 5
rng = np.random.default_rng(42)

interaction_mask = user_item_matrix_filled > 0
eligible_users = interaction_mask.sum(axis=1) >= 2

train_matrix = user_item_matrix_filled.copy()
test_items_by_user = {}

for user in train_matrix.index[eligible_users]:
    seen_items = train_matrix.columns[train_matrix.loc[user] > 0]
    held_out_item = rng.choice(seen_items.to_numpy())
    test_items_by_user[user] = {held_out_item}
    train_matrix.loc[user, held_out_item] = 0.0

train_similarity = safe_cosine_similarity(train_matrix)
train_similarity_df = pd.DataFrame(train_similarity, index=train_matrix.index, columns=train_matrix.index)

item_similarity = safe_cosine_similarity(train_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=train_matrix.columns, columns=train_matrix.columns)

def average_precision_at_k(recommended_items, relevant_items, k):
    if not relevant_items:
        return 0.0

    hits = 0
    score = 0.0
    for rank, item in enumerate(recommended_items[:k], start=1):
        if item in relevant_items:
            hits += 1
            score += hits / rank

    denom = min(len(relevant_items), k)
    return score / denom if denom > 0 else 0.0

def list_diversity(rec_list, item_sim_df):
    if len(rec_list) < 2:
        return np.nan

    distances = []
    for a, b in combinations(rec_list, 2):
        distances.append(1 - item_sim_df.loc[a, b])
    return float(np.mean(distances)) if distances else np.nan

users_evaluated = []
precision_scores = []
recall_scores = []
ap_scores = []
all_recommended_items = set()
diversity_scores = []

for user, relevant_items in test_items_by_user.items():
    recs = recommend_top_n(
        target_user=user,
        ratings_matrix=train_matrix,
        user_similarity_df=train_similarity_df,
        n=K,
        neighbor_top_m=10,
        min_neighbor_similarity=0.0,
        min_pred_score=0.0,
        already_rated_threshold=0,
        return_scores=False,
    )

    if len(recs) == 0:
        continue

    rec_set = set(recs)
    hits = len(rec_set & relevant_items)

    precision_scores.append(hits / K)
    recall_scores.append(hits / len(relevant_items))
    ap_scores.append(average_precision_at_k(recs, relevant_items, K))
    users_evaluated.append(user)

    all_recommended_items.update(recs)
    diversity_scores.append(list_diversity(recs, item_similarity_df))

num_items = train_matrix.shape[1]
coverage = len(all_recommended_items) / num_items if num_items > 0 else 0.0
mean_diversity = np.nanmean(diversity_scores) if len(diversity_scores) > 0 else np.nan

metrics = pd.DataFrame({
    'Metric': ['UsersEvaluated', 'Precision@K', 'Recall@K', 'MAP@K', 'Coverage', 'Diversity'],
    'Value': [
        len(users_evaluated),
        float(np.mean(precision_scores)) if precision_scores else 0.0,
        float(np.mean(recall_scores)) if recall_scores else 0.0,
        float(np.mean(ap_scores)) if ap_scores else 0.0,
        float(coverage),
        float(mean_diversity) if not np.isnan(mean_diversity) else np.nan,
    ],
})

metrics

,Metric,Value
0,UsersEvaluated,50.000000
1,Precision@K,0.036000
2,Recall@K,0.180000
3,MAP@K,0.109000
4,Coverage,0.760000
5,Diversity,0.804878


## Interpretation Guide
- **Precision@K**: Fraction of recommended items that are relevant.
- **Recall@K**: Fraction of relevant held-out items recovered in top-K.
- **MAP@K**: Rank-aware relevance quality across users.
- **Coverage**: Share of catalog items that appear in recommendations.
- **Diversity**: Average dissimilarity within each recommendation list.

Use this notebook's outputs as direct inputs for dashboarding and model comparison.